# 黄色边界线要素分离

本notebook用于将黄色边界分离为两条独立的线要素（LineString），生成ArcMap兼容的shapefile格式。

## 1. 导入必要的库

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
import json
import pandas as pd
import os
from shapely.geometry import LineString, Point
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 确保输出目录存在
output_dir = "output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建输出目录: {output_dir}")
else:
    print(f"输出目录已存在: {output_dir}")

In [ ]:
import matplotlib.font_manager as fm

font_path = "./PingFangSC-Medium.ttf"
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.sans-serif'] = ['PingFang SC']
    plt.rcParams['axes.unicode_minus'] = False
    print("已加载中文字体")
else:
    print("未找到字体文件，使用默认字体")

## 2. 定义边界线坐标

In [ ]:
# 定义两条边界线的起点和终点坐标
line1_coords = [(463, 176), (632, 377)]  # 边界线1
line2_coords = [(478, 428), (575, 770)]  # 边界线2

print("边界线坐标定义:")
print(f"边界线1: 起点{line1_coords[0]} → 终点{line1_coords[1]}")
print(f"边界线2: 起点{line2_coords[0]} → 终点{line2_coords[1]}")

# 计算线段长度
import math

def calculate_length(start, end):
    """计算两点间的欧几里得距离"""
    return math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)

line1_length = calculate_length(line1_coords[0], line1_coords[1])
line2_length = calculate_length(line2_coords[0], line2_coords[1])

print(f"\n线段长度:")
print(f"边界线1长度: {line1_length:.2f} 像素")
print(f"边界线2长度: {line2_length:.2f} 像素")

## 3. 创建线要素几何对象

In [ ]:
# 使用Shapely创建LineString几何对象
line1_geometry = LineString(line1_coords)
line2_geometry = LineString(line2_coords)

print("线要素几何对象创建完成:")
print(f"边界线1: {line1_geometry}")
print(f"边界线2: {line2_geometry}")

# 验证几何对象的有效性
print(f"\n几何对象有效性检查:")
print(f"边界线1有效: {'✓' if line1_geometry.is_valid else '✗'}")
print(f"边界线2有效: {'✓' if line2_geometry.is_valid else '✗'}")

# 获取边界框信息
print(f"\n边界框信息:")
print(f"边界线1边界框: {line1_geometry.bounds}")
print(f"边界线2边界框: {line2_geometry.bounds}")

## 4. 创建GeoDataFrame

In [ ]:
# 创建属性数据
data = {
    'line_id': [1, 2],
    'name': ['边界线1', '边界线2'],
    'start_x': [line1_coords[0][0], line2_coords[0][0]],
    'start_y': [line1_coords[0][1], line2_coords[0][1]],
    'end_x': [line1_coords[1][0], line2_coords[1][0]],
    'end_y': [line1_coords[1][1], line2_coords[1][1]],
    'length': [line1_geometry.length, line2_geometry.length],
    'geometry': [line1_geometry, line2_geometry]
}

# 创建GeoDataFrame
gdf = gpd.GeoDataFrame(data, crs='EPSG:4326')  # 使用WGS84坐标系，可根据需要调整

print("GeoDataFrame创建完成:")
print(gdf)
print(f"\n数据类型: {type(gdf)}")
print(f"坐标参考系统: {gdf.crs}")
print(f"要素数量: {len(gdf)}")

## 5. 可视化边界线

In [ ]:
# 创建可视化图表
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 左图：显示两条线的整体布局
gdf.plot(ax=ax1, color=['red', 'blue'], linewidth=3, alpha=0.8)
ax1.set_title('边界线整体布局', fontsize=14, fontweight='bold')
ax1.set_xlabel('X坐标 (像素)', fontsize=12)
ax1.set_ylabel('Y坐标 (像素)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend(['边界线1', '边界线2'], loc='best')

# 添加起点和终点标注
for idx, row in gdf.iterrows():
    color = 'red' if idx == 0 else 'blue'
    # 起点
    ax1.plot(row['start_x'], row['start_y'], 'o', color=color, markersize=8)
    ax1.annotate(f'起点{idx+1}\n({row["start_x"]}, {row["start_y"]})', 
                xy=(row['start_x'], row['start_y']), 
                xytext=(10, 10), textcoords='offset points',
                fontsize=10, ha='left')
    # 终点
    ax1.plot(row['end_x'], row['end_y'], 's', color=color, markersize=8)
    ax1.annotate(f'终点{idx+1}\n({row["end_x"]}, {row["end_y"]})', 
                xy=(row['end_x'], row['end_y']), 
                xytext=(10, -20), textcoords='offset points',
                fontsize=10, ha='left')

# 右图：属性信息表格
ax2.axis('off')
table_data = []
for idx, row in gdf.iterrows():
    table_data.append([
        row['name'],
        f"({row['start_x']}, {row['start_y']})",
        f"({row['end_x']}, {row['end_y']})",
        f"{row['length']:.2f}"
    ])

table = ax2.table(cellText=table_data,
                 colLabels=['线要素名称', '起点坐标', '终点坐标', '长度(像素)'],
                 cellLoc='center',
                 loc='center',
                 colWidths=[0.25, 0.25, 0.25, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
ax2.set_title('边界线属性信息', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# 打印详细统计信息
print("\n边界线统计信息:")
print(f"总长度: {gdf['length'].sum():.2f} 像素")
print(f"平均长度: {gdf['length'].mean():.2f} 像素")
print(f"最长线段: {gdf['length'].max():.2f} 像素")
print(f"最短线段: {gdf['length'].min():.2f} 像素")

## 6. 导出为Shapefile

In [ ]:
# 定义输出文件路径
shapefile_path = os.path.join(output_dir, "黄色边界线要素.shp")
geojson_path = os.path.join(output_dir, "黄色边界线要素.geojson")
csv_path = os.path.join(output_dir, "黄色边界线要素坐标.csv")
json_path = os.path.join(output_dir, "黄色边界线要素.json")
txt_path = os.path.join(output_dir, "黄色边界线要素坐标.txt")

# 导出为Shapefile (ArcMap兼容格式)
try:
    gdf.to_file(shapefile_path, driver='ESRI Shapefile', encoding='utf-8')
    print(f"✅ Shapefile导出成功: {shapefile_path}")
except Exception as e:
    print(f"❌ Shapefile导出失败: {e}")

# 导出为GeoJSON
try:
    gdf.to_file(geojson_path, driver='GeoJSON', encoding='utf-8')
    print(f"✅ GeoJSON导出成功: {geojson_path}")
except Exception as e:
    print(f"❌ GeoJSON导出失败: {e}")

# 导出坐标数据为CSV
try:
    # 创建坐标DataFrame
    coords_df = gdf[['line_id', 'name', 'start_x', 'start_y', 'end_x', 'end_y', 'length']].copy()
    coords_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ CSV坐标文件导出成功: {csv_path}")
except Exception as e:
    print(f"❌ CSV导出失败: {e}")

# 导出为JSON
try:
    # 转换为字典格式
    json_data = {
        "type": "FeatureCollection",
        "features": []
    }
    
    for idx, row in gdf.iterrows():
        feature = {
            "type": "Feature",
            "properties": {
                "line_id": int(row['line_id']),
                "name": row['name'],
                "start_x": int(row['start_x']),
                "start_y": int(row['start_y']),
                "end_x": int(row['end_x']),
                "end_y": int(row['end_y']),
                "length": round(row['length'], 2)
            },
            "geometry": {
                "type": "LineString",
                "coordinates": list(row['geometry'].coords)
            }
        }
        json_data["features"].append(feature)
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON文件导出成功: {json_path}")
except Exception as e:
    print(f"❌ JSON导出失败: {e}")

# 导出详细的TXT坐标文件
try:
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write("黄色边界线要素坐标详情\n")
        f.write("=" * 50 + "\n\n")
        
        for idx, row in gdf.iterrows():
            f.write(f"【{row['name']}】\n")
            f.write(f"线要素ID: {row['line_id']}\n")
            f.write(f"起点坐标: ({row['start_x']}, {row['start_y']})\n")
            f.write(f"终点坐标: ({row['end_x']}, {row['end_y']})\n")
            f.write(f"线段长度: {row['length']:.2f} 像素\n")
            f.write(f"几何类型: LineString\n")
            f.write(f"坐标序列: {list(row['geometry'].coords)}\n")
            f.write("-" * 30 + "\n\n")
        
        f.write(f"总计线要素数量: {len(gdf)}\n")
        f.write(f"总长度: {gdf['length'].sum():.2f} 像素\n")
        f.write(f"导出时间: {pd.Timestamp.now()}\n")
    
    print(f"✅ TXT坐标文件导出成功: {txt_path}")
except Exception as e:
    print(f"❌ TXT导出失败: {e}")

## 7. 验证输出文件

In [ ]:
print("文件验证结果:")
print("=" * 50)

# 检查生成的文件
output_files = [
    ("黄色边界线要素.shp", "Shapefile主文件"),
    ("黄色边界线要素.shx", "Shapefile索引文件"),
    ("黄色边界线要素.dbf", "Shapefile属性文件"),
    ("黄色边界线要素.cpg", "Shapefile编码文件"),
    ("黄色边界线要素.geojson", "GeoJSON文件"),
    ("黄色边界线要素坐标.csv", "CSV坐标文件"),
    ("黄色边界线要素.json", "JSON文件"),
    ("黄色边界线要素坐标.txt", "TXT坐标文件")
]

total_size = 0
for filename, description in output_files:
    file_path = os.path.join(output_dir, filename)
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        total_size += file_size
        print(f"✅ {filename:<30} ({description}) - {file_size:>6} bytes")
    else:
        print(f"❌ {filename:<30} ({description}) - 文件不存在")

print(f"\n📊 总文件大小: {total_size:,} bytes ({total_size/1024:.1f} KB)")

# 验证Shapefile完整性
shapefile_components = [".shp", ".shx", ".dbf"]
shapefile_complete = all(os.path.exists(os.path.join(output_dir, f"黄色边界线要素{ext}")) for ext in shapefile_components)

print(f"\n🗺️ Shapefile完整性: {'✅ 完整' if shapefile_complete else '❌ 不完整'}")

if shapefile_complete:
    print("\n📋 Shapefile组件:")
    for ext in [".shp", ".shx", ".dbf", ".cpg"]:
        file_path = os.path.join(output_dir, f"黄色边界线要素{ext}")
        if os.path.exists(file_path):
            size = os.path.getsize(file_path)
            print(f"   {ext:<4} 文件: {size:>6} bytes")

## 8. 总结与使用指南

In [ ]:
print("\n" + "="*80)
print("🎯 黄色边界线要素分离完成")
print("="*80)

print(f"\n📁 输出目录: {os.path.abspath(output_dir)}")
print(f"\n📂 生成的文件:")
line_output_files = [f for f in os.listdir(output_dir) if f.startswith("黄色边界线要素")]
for i, file in enumerate(sorted(line_output_files), 1):
    file_path = os.path.join(output_dir, file)
    file_size = os.path.getsize(file_path) / 1024  # KB
    print(f"   {i}. {file} ({file_size:.1f} KB)")

print(f"\n📊 线要素统计:")
print(f"   • 数量: {len(gdf)} 条线要素")
print(f"   • 边界线1长度: {gdf.iloc[0]['length']:.2f} 像素")
print(f"   • 边界线2长度: {gdf.iloc[1]['length']:.2f} 像素")
print(f"   • 总长度: {gdf['length'].sum():.2f} 像素")
print(f"   • 几何类型: LineString")

print(f"\n📍 坐标信息:")
for idx, row in gdf.iterrows():
    print(f"   • {row['name']}: ({row['start_x']}, {row['start_y']}) → ({row['end_x']}, {row['end_y']})")

print(f"\n🗺️ ArcMap导入指南:")
print(f"   1. 打开ArcMap")
print(f"   2. 点击 'Add Data' 按钮")
print(f"   3. 导航到: {os.path.abspath(output_dir)}")
print(f"   4. 选择: 黄色边界线要素.shp")
print(f"   5. 点击 'Add' 添加到地图")
print(f"   6. 线要素将显示为线条")
print(f"   7. 右键图层 → Properties → Symbology 设置线条样式")
print(f"   8. 右键图层 → Open Attribute Table 查看属性")

print(f"\n🔧 属性字段说明:")
print(f"   • line_id: 线要素唯一标识符")
print(f"   • name: 线要素名称")
print(f"   • start_x, start_y: 起点坐标")
print(f"   • end_x, end_y: 终点坐标")
print(f"   • length: 线段长度（像素单位）")

print(f"\n⚙️ 注意事项:")
print(f"   • 当前使用图像坐标系（像素坐标）")
print(f"   • 如需地理坐标，请在ArcMap中进行坐标转换")
print(f"   • 可通过地理配准将图像与实际地理位置对应")
print(f"   • 线要素适用于长度、方向等线性分析")

print(f"\n💡 使用建议:")
print(f"   • 用于测量和计算线性距离")
print(f"   • 适合网络分析和路径规划")
print(f"   • 可与其他要素进行空间叠加分析")
print(f"   • 支持缓冲区分析和邻近度计算")

if shapefile_complete:
    print(f"\n✅ 导出成功! 已生成ArcMap兼容的线要素文件")
else:
    print(f"\n❌ 导出警告! Shapefile可能不完整")